# Quantum-Enabled Grid Expansion Planning

*2026 Global Quantum + AI Challenge*

The energy transition demands intelligent grid expansion — integrating distributed
renewables, managing voltage constraints, and minimising transmission losses across
thousands of network nodes. This is an NP-hard combinatorial optimisation problem
where classical solvers hit computational walls as grid complexity grows.

This notebook demonstrates how uniqx solves grid planning as a **QUBO optimisation**
compiled to hybrid hardware through uniqx:

| Workload | Operation | Hardware affinity |
|:---------|:----------|:-----------------:|
| Grid topology optimisation | Ising eigendecomposition (`eigs`) | **QPU** |
| Network flow annealing | Simulated quantum annealing (`expv`) | **GPU / QPU** |
| Multi-feeder dispatch | VRP annealing with capacity constraints | **GPU** |
| Scaling analysis | Preflight cost model across grid sizes | **Cost model** |

**Challenge**: Can quantum optimisation enable real-time grid expansion planning?

Grid nodes map to the same QUBO formulation used for routing — uniqx handles
hardware selection while users focus on the energy network problem.

In [ ]:
import os
from uniqx import connect

# Default to a local service for development.
# For prod, export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
client = connect(endpoint)
print("Connected to", endpoint)
import time
import math
import matplotlib.pyplot as plt

from uniqx.domains.optimization.logistics import (
    DeliveryProblem,
    build_routing_module,
    build_vrp_module,
    build_annealing_module,
    decode_route,
)
from uniqx import parse_result
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Distribution Network Topologies

We model three distribution grid topologies as node layouts. Substations
and distribution nodes have geographic coordinates; Euclidean distances
approximate line impedances between nodes.

The `DeliveryProblem` class treats the first location as the **depot**
(primary substation) and remaining locations as distribution nodes.
"Stops" map to grid nodes, "distances" map to line impedances.

| Network | Nodes | Topology | Description |
|:--------|------:|:---------|:------------|
| Suburban | 7 | Radial | Tree-shaped feeder from central substation |
| Urban | 8 | Mesh | Dense interconnected urban distribution |
| Rural | 6 | Scattered | Long-distance scattered rural feeders |

In [ ]:
# --- Suburban radial: substation at origin, tree-shaped feeders ---
suburban = DeliveryProblem(locations=[
    (0.0, 0.0),   # primary substation
    (1.0, 0.0),   # feeder junction A
    (2.0, 0.0),   # distribution node
    (0.0, 1.0),   # feeder junction B
    (1.0, 1.0),   # distribution node
    (2.0, 1.0),   # distribution node
    (3.0, 1.0),   # end-of-line node
])

# --- Urban mesh: dense interconnected grid ---
urban = DeliveryProblem(locations=[
    (0.0, 0.0),   # primary substation
    (1.0, 0.0),   # junction
    (2.0, 0.0),   # junction
    (3.0, 0.0),   # distribution node
    (0.0, 1.0),   # junction
    (1.0, 1.0),   # distribution node
    (2.0, 1.0),   # distribution node
    (3.0, 1.0),   # end-of-line node
])

# --- Rural scattered: long-distance feeders ---
rural = DeliveryProblem.random_city(n_stops=6, seed=77)

grids = {
    "suburban_radial": suburban,
    "urban_mesh": urban,
    "rural_scattered": rural,
}

for name, grid in grids.items():
    D = grid.distance_matrix()
    total_impedance = sum(
        D[i][j] for i in range(grid.n_stops) for j in range(i + 1, grid.n_stops)
    )
    print(f"{name:>20}: {grid.n_stops} nodes, total line impedance = {total_impedance:.2f}")

## 2. Optimal Power Flow via QUBO

The minimum-loss power routing maps to a TSP-like QUBO: find the optimal
flow path through all grid nodes that minimises total impedance (distance).

The QUBO Hamiltonian encodes:
- **Constraint terms**: every node must be visited exactly once (power balance)
- **Objective terms**: minimise total line impedance along the flow path
- **Penalty parameter**: controls constraint satisfaction vs objective trade-off

uniqx compiles this to an Ising eigenvalue problem and routes to
the appropriate hardware (CPU, GPU, or QPU) based on matrix dimension.

In [ ]:
routing_modules = {}

for name, grid in grids.items():
    mod, inputs, meta = build_routing_module(grid, penalty=10.0, k=4)
    routing_modules[name] = (mod, inputs, meta)
    trunc = " (truncated)" if meta["truncated"] else ""
    print(
        f"{name:>20}: Ising dim={meta['dim']}, "
        f"n_nodes={meta['n_stops']}, k={meta['k']}{trunc}"
    )

## 3. Preflight: Hardware Comparison

uniqx's preflight endpoint returns execution options across
available hardware tiers. Each option includes estimated wall time,
cost, error rate, and carbon footprint — letting operators choose
the right trade-off for their grid planning scenario.

In [ ]:
def print_opts(label, options):
    print(f"\n--- {label} ---")
    if not options:
        return
    for o in options:
        f = " *" if o.get("recommended") else ""
        print(
            f"  {o['label']:<20} time={o['total_time']:>8.1f}"
            f"  cost=${o['total_cost_usd']:.4f}"
            f"  err={o['max_error_rate']:.4f}{f}"
        )


preflight_results = {}

for name, (mod, inputs, meta) in routing_modules.items():
    opts = preflight(mod, client=client)
    preflight_results[name] = opts
    print_opts(f"Grid: {name} ({meta['n_stops']} nodes, dim={meta['dim']})", opts)

## 4. Execute Grid Optimisation

We submit each grid topology to every available hardware option,
then decode the ground-state eigenvector into an optimal node
ordering (minimum-impedance flow path).

In [ ]:
tsp_results = {}

for name, (mod, inputs, meta) in routing_modules.items():
    opts = preflight_results[name]
    tsp_results[name] = {}

    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["eigenvalues", "eigenvectors"])
        evals = out.get("eigenvalues", ([], "", []))[2] if out else []
        evecs = out.get("eigenvectors", ([], "", []))[2] if out else []

        grid = grids[name]
        route = decode_route(evecs, grid.n_stops) if evecs else []
        ground_E = evals[0] if evals else 0.0

        tsp_results[name][opt["label"]] = {
            "time": wall,
            "eigenvalues": evals,
            "route": route,
            "ground_energy": ground_E,
            "option": opt,
        }
        print(
            f"  {name} / {opt['label']}: {wall:.2f}s, "
            f"E_0={ground_E:.4f}, flow_path={route}"
        )

## 5. Multi-Feeder Dispatch (VRP)

Grid expansion with multiple feeders is analogous to the Vehicle Routing
Problem: multiple feeders (vehicles) must serve all distribution nodes
from the primary substation (depot), subject to capacity constraints.

We use `build_vrp_module` with `n_vehicles=2` to model a dual-feeder
expansion scenario. The annealing schedule finds the lowest-energy
feeder assignment via repeated `expv` applications.

In [ ]:
vrp_results = {}

for name in ["suburban_radial", "urban_mesh"]:
    grid = grids[name]
    mod, inputs, meta = build_vrp_module(grid, n_vehicles=2)
    opts = preflight(mod, client=client)
    print_opts(f"VRP: {name} (2 feeders, dim={meta['dim']})", opts)

    vrp_results[name] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["final", "energy"])
        energy = out.get("energy", ([], "", [0.0]))[2][0] if out else 0.0

        vrp_results[name][opt["label"]] = {
            "time": wall,
            "energy": energy,
            "option": opt,
        }
        print(f"  {opt['label']}: {wall:.2f}s, feeder energy={energy:.4f}")

## 6. Grid Size Scaling

As distribution networks grow, the QUBO Hamiltonian dimension scales
quadratically with node count. We measure how preflight estimates
change across grid sizes to identify the crossover point where
GPU/QPU hardware becomes cost-effective over CPU-only execution.

In [ ]:
scaling = []

for n_nodes in [5, 6, 7, 8]:
    grid = DeliveryProblem.random_city(n_stops=n_nodes, seed=42)
    mod, inputs, meta = build_routing_module(grid)
    opts = preflight(mod, client=client)
    print(f"\nn_nodes={n_nodes} (Ising dim={meta['dim']}):")
    for opt in opts:
        f = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}"
            f"  cost=${opt['total_cost_usd']:.4f}{f}"
        )
        scaling.append({
            "n_nodes": n_nodes,
            "dim": meta["dim"],
            "label": opt["label"],
            "time": opt["total_time"],
            "cost": opt["total_cost_usd"],
            "recommended": opt.get("recommended", False),
        })

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}
colors_grid = {"suburban_radial": "#2563eb", "urban_mesh": "#16a34a", "rural_scattered": "#d97706"}
markers_grid = {"suburban_radial": "o", "urban_mesh": "s", "rural_scattered": "^"}

# --- Top-left: Grid topology plot ---
for name, grid in grids.items():
    locs = grid.locations
    xs = [l[0] for l in locs]
    ys = [l[1] for l in locs]
    c = colors_grid.get(name, "#6b7280")
    m = markers_grid.get(name, "o")
    axes[0, 0].scatter(xs, ys, s=80, marker=m, color=c, label=name, zorder=5)
    axes[0, 0].scatter([xs[0]], [ys[0]], s=160, marker="*", color="red", zorder=6)
    # Draw discovered flow path if available
    for hw_label, data in tsp_results.get(name, {}).items():
        route = data.get("route", [])
        if len(route) >= 2:
            for k in range(len(route)):
                a, b = route[k], route[(k + 1) % len(route)]
                if a < len(locs) and b < len(locs):
                    axes[0, 0].plot(
                        [locs[a][0], locs[b][0]],
                        [locs[a][1], locs[b][1]],
                        "--", alpha=0.35, color=c,
                    )
            break
axes[0, 0].set_xlabel("x (km)")
axes[0, 0].set_ylabel("y (km)")
axes[0, 0].set_title("Distribution Network Topologies")
axes[0, 0].legend(fontsize=7)
axes[0, 0].grid(alpha=0.3)

# --- Top-right: Execution time by grid topology (grouped bars) ---
layout_names = list(tsp_results.keys())
hw_labels_all = sorted(set(l for n in tsp_results for l in tsp_results[n]))
n_hw = len(hw_labels_all)
bar_width = 0.8 / max(n_hw, 1)
for hi, hw in enumerate(hw_labels_all):
    times = [tsp_results[n].get(hw, {}).get("time", 0) for n in layout_names]
    c = colors_hw.get(hw, "#6b7280")
    positions = [i + hi * bar_width for i in range(len(layout_names))]
    axes[0, 1].bar(positions, times, width=bar_width, color=c, label=hw, alpha=0.8)
axes[0, 1].set_xticks([i + bar_width * (n_hw - 1) / 2 for i in range(len(layout_names))])
axes[0, 1].set_xticklabels(layout_names, fontsize=7, rotation=15)
axes[0, 1].set_ylabel("Wall time (s)")
axes[0, 1].set_title("Optimal Power Flow: Execution by Topology")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(axis="y", alpha=0.3)

# --- Bottom-left: Scaling -- preflight time vs grid size (semilogy) ---
hw_labels_scale = sorted(set(d["label"] for d in scaling))
for hw in hw_labels_scale:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].semilogy(
        [d["n_nodes"] for d in sub],
        [d["time"] for d in sub],
        "o-", color=c, label=hw,
    )
axes[1, 0].set_xlabel("Grid nodes")
axes[1, 0].set_ylabel("Estimated time (log)")
axes[1, 0].set_title("Scaling: Preflight Time vs Grid Size")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# --- Bottom-right: Scaling -- cost vs grid size ---
for hw in hw_labels_scale:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].plot(
        [d["n_nodes"] for d in sub],
        [d["cost"] for d in sub],
        "s-", color=c, label=hw,
    )
axes[1, 1].set_xlabel("Grid nodes")
axes[1, 1].set_ylabel("Estimated cost ($)")
axes[1, 1].set_title("Scaling: Cost vs Grid Size")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle(
    "Grid Expansion Planning on Hybrid Hardware",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()

## Summary

| Problem | Encoding | Ising dim | Best hardware |
|:--------|:---------|:----------|:--------------|
| Suburban radial (7 nodes) | QUBO → Ising | 49 | CPU |
| Urban mesh (8 nodes) | QUBO → Ising (truncated) | 64→256 | GPU |
| Rural scattered (6 nodes) | QUBO → Ising | 36 | CPU |
| Multi-feeder VRP (2 feeders) | QUBO + capacity | same as TSP | GPU |
| Scaling (5–8 nodes) | QUBO → Ising | 25–256 | CPU → GPU crossover |

The QUBO → Ising encoding maps grid expansion planning onto quantum
Hamiltonians. Distribution nodes become stops in a combinatorial
optimisation problem; line impedances become edge weights. The
uniqx's cost model routes small grids (< 50 nodes) to CPU and
larger networks to GPU/QPU, enabling real-time planning decisions
at distribution-scale complexity.

Multi-feeder dispatch (VRP) extends the single-path optimisation
to realistic scenarios where multiple feeders share the load from
a primary substation — a direct requirement for utility distribution
network modernisation.